## Cell 1 — GPU Check
Make sure Colab is set to GPU runtime: Runtime → Change runtime type → T4 GPU

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU")

CUDA available: True
Device: Tesla T4


## Cell 2 — Install dependencies
PyTorch and transformers are pre-installed on Colab. Only xgboost and shap need installing.

In [2]:
!pip install -q xgboost shap

## Cell 3 — Mount Google Drive
Used to save/load expensive intermediate results (embeddings, sentiment scores) so you don't recompute on kernel restart.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/cs1671_stock'
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive directory ready:", DRIVE_DIR)

Mounted at /content/drive
Drive directory ready: /content/drive/MyDrive/cs1671_stock


## Cell 4 — Upload data file
Upload `sp500_headlines_with_prices sep ~.csv` to Colab using the cell below, or copy it to your Drive and adjust the path.

In [4]:
from google.colab import files
# Option A: upload manually (small file, easy)
uploaded = files.upload()  # uncomment to use

# Option B: copy from Drive (if you already put it there)
# !cp "/content/drive/MyDrive/sp500_headlines_with_prices sep ~.csv" .

# Verify file exists
import os
DATA_FILE = "sp500_headlines_with_prices sep ~.csv"
print("File found:", os.path.exists(DATA_FILE))

Saving sp500_headlines_with_prices sep ~.csv to sp500_headlines_with_prices sep ~.csv
File found: True


## Cell 5 — Imports

In [5]:
import pandas as pd
import numpy as np
import re
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_name = "ProsusAI/finbert"

np.random.seed(42)
torch.manual_seed(42)

Using device: cuda


## Cell 6 — Load FinBERT models

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

finbert_model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
finbert_model.eval()

# Reuse finbert's BERT backbone instead of loading a second copy
bert_model = finbert_model.bert

print("Label mapping:", finbert_model.config.id2label)
# Expected: {0: 'positive', 1: 'negative', 2: 'neutral'}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Label mapping: {0: 'positive', 1: 'negative', 2: 'neutral'}


## Cell 7 — Load data & train/test split

In [7]:
df = pd.read_csv(DATA_FILE)

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# No pre-shift: use today's data as features, predict tonight's overnight gap
# Target: next open > today's close (same as StockPredict.ipynb)
df["target"] = (df["open_price"].shift(-1) > df["close_price"]).astype(int)

df["headline_list"] = df["combined_headline"].fillna("").apply(
    lambda x: x.split("~") if isinstance(x, str) else []
)

df["news_t"]    = df["combined_headline"].fillna("")
df["news_lag1"] = df["news_t"].shift(1).fillna("")
df["news_lag2"] = df["news_t"].shift(2).fillna("")
df["news_lag3"] = df["news_t"].shift(3).fillna("")

# Chronological split — no shuffle for time series
split_index = int(len(df) * 0.8)
train_df = df.iloc[:split_index].copy().reset_index(drop=True)
test_df  = df.iloc[split_index:].copy().reset_index(drop=True)

print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")
print("Target distribution (train):", train_df["target"].value_counts().to_dict())

Train: 2805 rows | Test: 702 rows
Target distribution (train): {1: 1475, 0: 1330}


## Cell 8 — CLS embeddings (FinBERT backbone)
Saves result to Drive. If already computed, loads from Drive and skips.

In [8]:
TRAIN_CLS_PATH = os.path.join(DRIVE_DIR, 'train_cls_v2.npy')
TEST_CLS_PATH  = os.path.join(DRIVE_DIR, 'test_cls_v2.npy')

def get_cls_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = bert_model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

if os.path.exists(TRAIN_CLS_PATH) and os.path.exists(TEST_CLS_PATH):
    print("Loading CLS embeddings from Drive (v2)...")
    train_cls_raw = np.load(TRAIN_CLS_PATH)
    test_cls_raw  = np.load(TEST_CLS_PATH)
else:
    print("Computing CLS embeddings (this takes a few minutes)...")
    # today's news + yesterday's news — aligned with target (predict tonight's gap)
    train_text = (train_df["news_t"].fillna("") + " " + train_df["news_lag1"].fillna("")).tolist()
    test_text  = (test_df["news_t"].fillna("")  + " " + test_df["news_lag1"].fillna("")).tolist()
    train_cls_raw = get_cls_embeddings(train_text)
    test_cls_raw  = get_cls_embeddings(test_text)
    np.save(TRAIN_CLS_PATH, train_cls_raw)
    np.save(TEST_CLS_PATH,  test_cls_raw)
    print("Saved to Drive.")

pca = PCA(n_components=12)
train_cls = pca.fit_transform(train_cls_raw)
test_cls  = pca.transform(test_cls_raw)

cls_train_df = pd.DataFrame(train_cls, columns=[f"emb_{i}" for i in range(12)])
cls_test_df  = pd.DataFrame(test_cls,  columns=[f"emb_{i}" for i in range(12)])

train_df = pd.concat([train_df.reset_index(drop=True), cls_train_df], axis=1)
test_df  = pd.concat([test_df.reset_index(drop=True),  cls_test_df],  axis=1)
print("CLS embeddings added. Train shape:", train_df.shape)

Computing CLS embeddings (this takes a few minutes)...
Saved to Drive.
CLS embeddings added. Train shape: (2805, 26)


## Cell 9 — TF-IDF features (disabled)
TF-IDF removed: 5000 sparse dims hurt performance by drowning out FinBERT signal.

In [9]:
# tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
#
# train_text_tfidf = train_df["news_t"] + " " + train_df["news_lag1"]
# test_text_tfidf  = test_df["news_t"]  + " " + test_df["news_lag1"]
#
# tfidf_train = tfidf.fit_transform(train_text_tfidf).toarray()
# tfidf_test  = tfidf.transform(test_text_tfidf).toarray()
#
# n_tfidf = tfidf_train.shape[1]
# tfidf_train_df = pd.DataFrame(tfidf_train, columns=[f"tfidf_{i}" for i in range(n_tfidf)])
# tfidf_test_df  = pd.DataFrame(tfidf_test,  columns=[f"tfidf_{i}" for i in range(n_tfidf)])
#
# train_df = pd.concat([train_df, tfidf_train_df], axis=1)
# test_df  = pd.concat([test_df,  tfidf_test_df],  axis=1)
# print(f"TF-IDF added ({n_tfidf} features). Train shape:", train_df.shape)

## Cell 10 — FinBERT sentiment scores
Saves result to Drive. If already computed, loads from Drive and skips.

In [10]:
TRAIN_SENT_PATH = os.path.join(DRIVE_DIR, 'train_sentiment_v2.csv')
TEST_SENT_PATH  = os.path.join(DRIVE_DIR, 'test_sentiment_v2.csv')

SENT_COLS = ["sent_neg_mean", "sent_neu_mean", "sent_pos_mean",
             "sent_neg_max", "sent_pos_max", "sent_volatility", "sent_polarity"]

def get_sentiment_batch(texts, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = finbert_model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)
        all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)

def compute_sentiment(df):
    neg_mean, neu_mean, pos_mean = [], [], []
    neg_max, pos_max, sent_vol = [], [], []
    for headlines in df["headline_list"]:
        headlines = [h for h in headlines if isinstance(h, str) and h.strip()]
        if len(headlines) == 0:
            headlines = ["neutral market"]
        sentiments = get_sentiment_batch(headlines)
        # FinBERT label mapping: {0: positive, 1: negative, 2: neutral}
        pos = sentiments[:, 0]
        neg = sentiments[:, 1]
        neu = sentiments[:, 2]
        pos_mean.append(np.mean(pos))
        neg_mean.append(np.mean(neg))
        neu_mean.append(np.mean(neu))
        neg_max.append(np.max(neg))
        pos_max.append(np.max(pos))
        sent_vol.append(np.std(pos - neg))
    df["sent_neg_mean"]   = neg_mean
    df["sent_neu_mean"]   = neu_mean
    df["sent_pos_mean"]   = pos_mean
    df["sent_neg_max"]    = neg_max
    df["sent_pos_max"]    = pos_max
    df["sent_volatility"] = sent_vol
    df["sent_polarity"]   = df["sent_pos_mean"] - df["sent_neg_mean"]
    return df

if os.path.exists(TRAIN_SENT_PATH) and os.path.exists(TEST_SENT_PATH):
    print("Loading sentiment scores from Drive (v2)...")
    train_sent = pd.read_csv(TRAIN_SENT_PATH)
    test_sent  = pd.read_csv(TEST_SENT_PATH)
    for col in SENT_COLS:
        train_df[col] = train_sent[col].values
        test_df[col]  = test_sent[col].values
else:
    print("Computing sentiment (~10-15 min on GPU)...")
    train_df = compute_sentiment(train_df)
    test_df  = compute_sentiment(test_df)
    train_df[SENT_COLS].to_csv(TRAIN_SENT_PATH, index=False)
    test_df[SENT_COLS].to_csv(TEST_SENT_PATH,   index=False)
    print("Saved to Drive.")

print("Sentiment columns added.")

Computing sentiment (~10-15 min on GPU)...
Saved to Drive.
Sentiment columns added.


## Cell 11 — Sentiment lags + feature engineering

In [11]:
sent_lag_cols_base = ["sent_neg_mean", "sent_neu_mean", "sent_pos_mean",
                      "sent_polarity", "sent_volatility"]

for df_ in [train_df, test_df]:
    for lag in [1, 2]:
        for col in sent_lag_cols_base:
            df_[f"{col}_lag{lag}"] = df_[col].shift(lag)
    lag_cols = [f"{c}_lag{l}" for l in [1, 2] for c in sent_lag_cols_base]
    df_[lag_cols] = df_[lag_cols].fillna(0)

def get_market_signals(text):
    text = str(text).lower()
    patterns = {
        "flag_index": r"\b(s&p|s&p 500|sp500|spx|spy|voo|gspc|market)\b",
        "flag_fed":   r"\b(fed|federal reserve|powell|fomc|interest rate(s)?|hawkish|dovish|yield(s)?)\b",
        "flag_macro": r"\b(cpi|ppi|inflation|gdp|unemployment|nfp|retail sales|pce)\b",
        "flag_mag7":  r"\b(aapl|apple|msft|microsoft|nvda|nvidia|goog|google|amzn|amazon|meta|tsla|tesla)\b",
        "flag_risk":  r"\b(war|tariff(s)?|sanction(s)?|recession|crisis|election|default|vix|geopolitical)\b"
    }
    return pd.Series({flag: int(bool(re.search(pat, text))) for flag, pat in patterns.items()})

for df_ in [train_df, test_df]:
    signals = df_["combined_headline"].apply(get_market_signals)
    for col in signals.columns:
        df_[col] = signals[col].values

print("Feature engineering done.")

Feature engineering done.


## Cell 12 — NLP-only feature selection & Logistic Regression

In [12]:
# NLP-only features: CLS embeddings + sentiment scores + sentiment lags + market flags
nlp_feature_cols = (
    [f"emb_{i}" for i in range(12)] +
    ["sent_neg_mean", "sent_neu_mean", "sent_pos_mean",
     "sent_neg_max", "sent_pos_max", "sent_volatility", "sent_polarity"] +
    [f"{c}_lag{l}" for l in [1, 2] for c in sent_lag_cols_base] +
    ["flag_index", "flag_fed", "flag_macro", "flag_mag7", "flag_risk"]
)

X_train = train_df[nlp_feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = test_df[nlp_feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train = train_df["target"]
y_test  = test_df["target"]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Features: {len(nlp_feature_cols)} total "
      f"(12 CLS + 7 sentiment + 10 sentiment lags + 5 flags)")

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_model.fit(X_train_s, y_train)
print("Logistic Regression training done.")

X_train: (2805, 34) | X_test: (702, 34)
Features: 34 total (12 CLS + 7 sentiment + 10 sentiment lags + 5 flags)
Logistic Regression training done.


## Cell 13 — Results

In [13]:
test_preds = lr_model.predict(X_test_s)
test_probs = lr_model.predict_proba(X_test_s)[:, 1]

print(classification_report(y_test, test_preds))
print("\n================ FINAL TEST RESULTS ================")
print("Accuracy:", accuracy_score(y_test, test_preds))
print("ROC-AUC:", roc_auc_score(y_test, test_probs))

              precision    recall  f1-score   support

           0       0.41      0.02      0.04       326
           1       0.53      0.97      0.69       376

    accuracy                           0.53       702
   macro avg       0.47      0.50      0.37       702
weighted avg       0.48      0.53      0.39       702


================ FINAL TEST RESULTS ================
Accuracy: 0.5313390313390314
ROC-AUC: 0.4706059913849367


## Cell 14 — Feature importance (Logistic Regression coefficients)

In [14]:
coef_df = pd.DataFrame({
    "feature":     nlp_feature_cols,
    "coefficient": lr_model.coef_[0]
}).assign(abs_coef=lambda d: d["coefficient"].abs()
).sort_values("abs_coef", ascending=False)

print("===== TOP 20 FEATURES BY |COEFFICIENT| =====")
print(coef_df[["feature", "coefficient"]].head(20).to_string(index=False))

===== TOP 20 FEATURES BY |COEFFICIENT| =====
             feature  coefficient
  sent_neu_mean_lag1    -0.466606
  sent_neg_mean_lag1    -0.376637
  sent_pos_mean_lag1    -0.356205
  sent_pos_mean_lag2     0.155004
  sent_neu_mean_lag2     0.141622
  sent_neg_mean_lag2     0.127516
  sent_polarity_lag1     0.085679
              emb_10     0.076767
sent_volatility_lag1     0.071642
        sent_pos_max     0.065633
       sent_neu_mean     0.065274
               emb_5    -0.054568
          flag_index     0.041978
       sent_neg_mean    -0.040984
       sent_pos_mean    -0.040712
               emb_3     0.030255
               emb_0    -0.030040
               emb_4    -0.028514
               emb_7     0.028306
        sent_neg_max     0.027643


## Analysis: Why Accuracy Remains Low (53.1%)

**Model behavior**
- Class 0 recall = 0.02 — model almost never predicts "down"
- Class 1 recall = 0.97 — model defaults to predicting "up" nearly always
- ROC-AUC = 0.47 — probability estimates are worse than random, effectively inverted

**Root cause: CLS embeddings lack discriminative signal**
- CLS token encodes general sentence semantics, not financial sentiment direction
- FinBERT's direct 3-dim output (positive/negative/neutral probabilities) is purpose-trained on financial text — a much cleaner signal
- Compressing 768-dim CLS to 12-dim via PCA further dilutes whatever signal exists

**Feature design mismatch**
- Adding sentiment lags, derived metrics, and keyword flags adds noise without adding information
- Top features by coefficient are all `sent_xxx_lag1` — yesterday's sentiment is too indirect for predicting tonight's overnight gap
- 34 features across 2805 samples gives Logistic Regression too many dimensions relative to signal strength

**Comparison with StockPredict.ipynb (64.1%)**
- StockPredict uses only 3 features: raw FinBERT pos/neg/neu probabilities
- LinearSVC finds a cleaner decision boundary in that low-dimensional, high-signal space
- More features ≠ better performance when the added features carry no predictive signal